In [1]:
import pandas as pd
from openai import OpenAI
import os
from src.config import QDRANT_URL, QDRANT_API_KEY, OPENAI_API_KEY, OPENAI_MODEL, OPENAI_API_URL,\
     DENSE_EMBEDDING_MODEL_PATH, OPENAI_MODEL_MINI, INTERIM_DATA_DIR, PROCESSED_DATA_DIR, DEVICE, \
     DEEPSEEK_MODEL
from src.dataset import render_table_ddls, rewrite_query_descriptions_csv, get_sqlite_database_path
from src.script_generator import (
    generate_query_descriptions,
    generate_related_query_descriptions_csv,
    generate_sql_scripts_and_results,
)
from src.vanna_connector import initialize_vanna
from src.search_metrics import run_search_evaluation_pipeline, compute_ranking_metrics
from src.generation_metrics import run_generation_evaluation_pipeline, compute_generation_metrics

/home/user/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
url = os.path.join(PROCESSED_DATA_DIR, "sakila", "sakila_inline_short.sqlite.db")
DATABASE_NAME = "sakila"


sqlite_config = {
    "params": {
        "url": str(url) 
    },
    "type": "sqlite"}

qdrant_config = {"fastembed_model": DENSE_EMBEDDING_MODEL_PATH,
                 "url": QDRANT_URL, 
                 "api_key": QDRANT_API_KEY,
                 "device": DEVICE}

openai_config = {"api_key": OPENAI_API_KEY,
                 "model": OPENAI_MODEL,
                 "base_url": OPENAI_API_URL}

In [3]:
vanna_client = initialize_vanna(db_config=sqlite_config,
                                qdrant_config=qdrant_config,
                                openai_config=openai_config)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2426.63it/s]
/home/user/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/vanna/legacy/qdrant/qdrant.py:49: UserWarning: Api key is used with an insecure connection.
  self._client = QdrantClient(


# Adding DDL to vector store

In [10]:
table_ddls = render_table_ddls(
    database_name=DATABASE_NAME,
    comment_style="inline",
    comment_variant="short",
)

print(table_ddls[0])

CREATE TABLE act ( -- Актёры фильмов.
  a01 numeric NOT NULL, -- Идентификатор актёра.
  a02 VARCHAR(45) NOT NULL, -- Имя актёра.
  a03 VARCHAR(45) NOT NULL, -- Фамилия актёра.
  a04 TIMESTAMP NOT NULL, -- Дата изменения записи.
  PRIMARY KEY (a01)
);


In [ ]:
# # save as example for inference
# df = pd.DataFrame({"ddl": table_ddls})
# df.to_csv(f"{INTERIM_DATA_DIR}/{DATABASE_NAME}/table_ddls_inline_short.csv", index=False)

In [14]:
for table_ddl in table_ddls:
    vanna_client.train(ddl=table_ddl)

Adding ddl: CREATE TABLE act ( -- Актёры фильмов.
  a01 numeric NOT NULL, -- Идентификатор актёра.
  a02 VARCHAR(45) NOT NULL, -- Имя актёра.
  a03 VARCHAR(45) NOT NULL, -- Фамилия актёра.
  a04 TIMESTAMP NOT NULL, -- Дата изменения записи.
  PRIMARY KEY (a01)
);
Adding ddl: CREATE TABLE cnt ( -- Страны.
  c01 SMALLINT NOT NULL, -- Идентификатор страны.
  c02 VARCHAR(50) NOT NULL, -- Название страны.
  c03 TIMESTAMP, -- Дата изменения записи.
  PRIMARY KEY (c01)
);
Adding ddl: CREATE TABLE cty ( -- Города.
  d01 int NOT NULL, -- Идентификатор города.
  d02 VARCHAR(50) NOT NULL, -- Название города.
  d03 SMALLINT NOT NULL, -- Идентификатор страны.
  d04 TIMESTAMP NOT NULL, -- Дата изменения записи.
  PRIMARY KEY (d01),
  CONSTRAINT fk_cty_cnt FOREIGN KEY (d03) REFERENCES cnt (c01) ON DELETE NO ACTION ON UPDATE CASCADE
);
Adding ddl: CREATE TABLE adr ( -- Адреса.
  e01 int NOT NULL, -- Идентификатор адреса.
  e02 VARCHAR(50) NOT NULL, -- Адрес (строка 1).
  e03 VARCHAR(50) DEFAULT NULL, 

# Generating artificial query description -> SQL queries -> creating descriptions in different format -> adding them to vectore store

In [2]:
openai_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_URL,
)

In [ ]:
descriptions_csv_path, schema_description_path, sqlite_db_path = generate_query_descriptions(
    client=openai_client,
    model=OPENAI_MODEL,
    comment_style="inline",
    comment_variant="short",
    database_name=DATABASE_NAME,
    counts_by_difficulty={"easy": 20, "medium": 20, "hard": 20},
    temperature=1.0,
)

print("Descriptions CSV:", descriptions_csv_path)
print("Schema description:", schema_description_path)
print("SQLite DB:", sqlite_db_path)

Descriptions CSV: /home/user/cursor_projects/vanna-sql/data/interim/sakila/query_descriptions_inline_short.csv
Schema description: /home/user/cursor_projects/vanna-sql/data/interim/sakila/schema_description_inline_short.txt
SQLite DB: /home/user/cursor_projects/vanna-sql/data/processed/sakila/sakila_inline_short.sqlite.db


In [16]:
generation_summary = generate_sql_scripts_and_results(
    client=openai_client,
    model=OPENAI_MODEL,
    sqlite_db_path=sqlite_db_path,
    interim_dir=descriptions_csv_path.parent,
    descriptions_csv_path=descriptions_csv_path,
    schema_description_path=schema_description_path,
    temperature=0.2,
)

generation_summary

{'total': 60,
 'generated': 60,
 'saved': 56,
 'failed': 2,
 'empty': 2,
 'invalid_sql': 0}

In [17]:
# descriptions_csv_path = os.path.join(INTERIM_DATA_DIR, "bank_transaction_monitoring", "query_descriptions_inline_short.csv")
# schema_description_path = os.path.join(INTERIM_DATA_DIR, "bank_transaction_monitoring", "schema_description_inline_short.txt")


rewritten_descriptions_csv_path = rewrite_query_descriptions_csv(
    descriptions_csv_path=descriptions_csv_path,
    client=openai_client,
    model=OPENAI_MODEL_MINI,
    schema_description_path=schema_description_path,
    rewrite_styles=("short", "business", "technical"),
    source_column="query",
    temperature=0.9,
)

print("Rewritten descriptions CSV:", rewritten_descriptions_csv_path)

Rewritten descriptions CSV: /home/user/cursor_projects/vanna-sql/data/interim/sakila/query_descriptions_inline_short_rewritten.csv


In [ ]:
# schema_description_path = os.path.join(INTERIM_DATA_DIR, "bank_transaction_monitoring", "schema_description_inline_short.txt")
# rewritten_descriptions_csv_path = os.path.join(INTERIM_DATA_DIR, "bank_transaction_monitoring", "query_descriptions_inline_short_rewritten.csv")

related_descriptions_csv_path = generate_related_query_descriptions_csv(
    rewritten_descriptions_csv_path=rewritten_descriptions_csv_path,
    schema_description_path=schema_description_path,
    client=openai_client,
    model=DEEPSEEK_MODEL,
    rewrite_styles=("short", "business", "technical"),
    source_column="query",
    temperature=0.9,
    num_queries=3,
)

print("Related descriptions CSV:", related_descriptions_csv_path)

Generating related query descriptions: 100%|██████████| 540/540 [2:14:04<00:00, 14.90s/it]  

Related descriptions CSV: /home/user/cursor_projects/vanna-sql/data/interim/bank_transaction_monitoring/query_descriptions_inline_short_rewritten_related.csv


In [4]:
DATABASE_NAME = "sakila"
related_descriptions_csv_path = '/home/user/cursor_projects/vanna-sql/data/interim/sakila/query_descriptions_inline_short_rewritten_related.csv'

In [ ]:
search_result_paths = run_search_evaluation_pipeline(
    related_descriptions_csv_path=related_descriptions_csv_path,
    interim_dir=INTERIM_DATA_DIR,
    database_name=DATABASE_NAME,
    qdrant_config=qdrant_config,
    openai_config=openai_config,
    db_config=sqlite_config,
    description_styles=("short", "business", "technical"),
    search_query_columns=None,
    n_results=5,
    comment_style="inline",
    comment_variant="short",
)

print("Search results:", search_result_paths)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2710.02it/s]
/home/user/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/vanna/legacy/qdrant/qdrant.py:49: UserWarning: Api key is used with an insecure connection.
  self._client = QdrantClient(
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2391.43it/s]
/home/user/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/vanna/legacy/qdrant/qdrant.py:49: UserWarning: Api key is used with an insecure connection.
  self._client = QdrantClient(
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4525.91it/s]
/home/user/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/vanna/legacy/qdrant/qdrant.py:49: UserWarning: Api key is used with an insecure connection.
  self._client = QdrantClient(
Searching (technical): 100%|██████████| 504/504 [00:14<00:00, 35.81it/s]

Search results: [PosixPath('/home/user/cursor_projects/vanna-sql/data/interim/sakila/search/short_top_5.csv'), PosixPath('/home/user/cursor_projects/vanna-sql/data/interim/sakila/search/business_top_5.csv'), PosixPath('/home/user/cursor_projects/vanna-sql/data/interim/sakila/search/technical_top_5.csv')]
Metrics: [PosixPath('/home/user/cursor_projects/vanna-sql/data/interim/sakila/search/ranking_metrics_short.csv'), PosixPath('/home/user/cursor_projects/vanna-sql/data/interim/sakila/search/ranking_metrics_business.csv'), PosixPath('/home/user/cursor_projects/vanna-sql/data/interim/sakila/search/ranking_metrics_technical.csv')]


In [ ]:
metrics_paths = compute_ranking_metrics(
    search_result_paths=search_result_paths,
    k=5,
)

print("Metrics:", metrics_paths)

In [6]:
for path in metrics_paths:
    print(path.name)
    display(pd.read_csv(path))

ranking_metrics_short.csv


,metric,column,k,value
0,mrr,predicted_short,5,0.535218
1,map_at_k,predicted_short,5,0.535218
2,recall_at_k,predicted_short,5,0.761905
3,mrr,predicted_business,5,0.578571
4,map_at_k,predicted_business,5,0.578571
5,recall_at_k,predicted_business,5,0.773810
6,mrr,predicted_technical,5,0.478175
7,map_at_k,predicted_technical,5,0.478175
8,recall_at_k,predicted_technical,5,0.744048
9,mrr,combined,5,0.562871


ranking_metrics_business.csv


,metric,column,k,value
0,mrr,predicted_short,5,0.516964
1,map_at_k,predicted_short,5,0.516964
2,recall_at_k,predicted_short,5,0.767857
3,mrr,predicted_business,5,0.522421
4,map_at_k,predicted_business,5,0.522421
5,recall_at_k,predicted_business,5,0.773810
6,mrr,predicted_technical,5,0.544940
7,map_at_k,predicted_technical,5,0.544940
8,recall_at_k,predicted_technical,5,0.815476
9,mrr,combined,5,0.546478


ranking_metrics_technical.csv


,metric,column,k,value
0,mrr,predicted_short,5,0.508135
1,map_at_k,predicted_short,5,0.508135
2,recall_at_k,predicted_short,5,0.767857
3,mrr,predicted_business,5,0.544544
4,map_at_k,predicted_business,5,0.544544
5,recall_at_k,predicted_business,5,0.773810
6,mrr,predicted_technical,5,0.498710
7,map_at_k,predicted_technical,5,0.498710
8,recall_at_k,predicted_technical,5,0.785714
9,mrr,combined,5,0.539029


In [ ]:
rewritten_descriptions_csv_path = INTERIM_DATA_DIR / DATABASE_NAME / "query_descriptions_inline_short_rewritten.csv"

predicted_dirs = run_generation_evaluation_pipeline(
    descriptions_csv_path=rewritten_descriptions_csv_path,
    interim_dir=INTERIM_DATA_DIR,
    database_name=DATABASE_NAME,
    models={
        "mini": OPENAI_MODEL_MINI,
        "gpt5": OPENAI_MODEL,
    },
    qdrant_config=qdrant_config,
    openai_config=openai_config,
    comment_styles=("inline", ),
    description_styles=("short", "business",),
    db_config=sqlite_config,
    n_folds=5,
)

print("Predicted result dirs:", predicted_dirs)

In [7]:
ground_truth_results_dir = INTERIM_DATA_DIR / DATABASE_NAME / "results"

generation_metrics_paths = compute_generation_metrics(
    ground_truth_results_dir=ground_truth_results_dir,
    predicted_results_dirs=predicted_dirs,
    descriptions_csv_path=rewritten_descriptions_csv_path,
)

for path in generation_metrics_paths:
    print(path.name)
    df = pd.read_csv(path)
    display(df.groupby("difficulty")['label'].value_counts())

inline_short_mini.csv


difficulty  label                               
easy        Everything matches                      9
            Not matching values                     9
            Not matching number of columns          2
hard        Not matching number of rows and cols    9
            Not generated                           4
            Not matching number of columns          2
            Not matching number of rows             1
medium      Not matching number of columns          9
            Not matching values                     5
            Not generated                           3
            Everything matches                      2
            Not matching number of rows             1
Name: count, dtype: int64

inline_business_mini.csv


difficulty  label                               
easy        Everything matches                      11
            Not matching values                      7
            Not matching number of columns           2
hard        Not generated                            8
            Not matching number of rows and cols     7
            Not matching number of columns           1
medium      Not matching number of columns           5
            Not matching values                      5
            Everything matches                       3
            Not generated                            3
            Not matching number of rows              2
            Not matching number of rows and cols     2
Name: count, dtype: int64

inline_short_gpt5.csv


difficulty  label                               
easy        Everything matches                      11
            Not matching values                      7
            Not matching number of columns           2
hard        Not matching number of rows and cols     6
            Not matching number of columns           4
            Not generated                            3
            Not matching number of rows              3
medium      Not matching values                      9
            Not generated                            4
            Everything matches                       3
            Not matching number of columns           3
            Not matching number of rows and cols     1
Name: count, dtype: int64

inline_business_gpt5.csv


difficulty  label                               
easy        Everything matches                      14
            Not matching values                      4
            Not matching number of columns           2
hard        Not matching number of rows and cols     8
            Not generated                            6
            Not matching number of columns           2
medium      Not matching number of columns           7
            Not matching values                      6
            Not generated                            3
            Not matching number of rows              2
            Everything matches                       1
            Not matching number of rows and cols     1
Name: count, dtype: int64

In [17]:
df[df["label"] == "Not matching values"]

,uuid,difficulty,label
1,1f20ef88-9279-4248-9373-dd67f628e895,easy,Not matching values
2,58d71401-2b79-4c01-aa27-d796ade816e6,easy,Not matching values
4,0b31dac1-bbe8-4a24-b1f4-17d4289fcb7c,easy,Not matching values
6,71c28807-ed2e-45ae-92e3-f28bf73e3332,easy,Not matching values
7,5f5b0bf9-0d88-4a6d-b2ff-75c698b26343,easy,Not matching values
11,7dd87e0e-1a56-4abc-855d-dc7f5dc78799,easy,Not matching values
12,c44e6641-c6a1-48ce-9032-2f72be70c8cf,easy,Not matching values
20,3aea5482-5e8a-42be-9898-216a936a48ab,medium,Not matching values
23,605ee2cc-3143-4d32-ba7c-f4ad88b950fd,medium,Not matching values
25,ad40a2da-f48c-4d48-ad2d-c6f833a55201,medium,Not matching values


In [10]:
url = os.path.join(PROCESSED_DATA_DIR, "sakila", "sakila_inline_short.sqlite.db")
DATABASE_NAME = "sakila"


sqlite_config = {
    "params": {
        "url": str(url) 
    },
    "type": "sqlite"}

qdrant_config = {"fastembed_model": DENSE_EMBEDDING_MODEL_PATH,
                 "url": QDRANT_URL, 
                 "api_key": QDRANT_API_KEY,
                 "device": DEVICE}

openai_config = {"api_key": OPENAI_API_KEY,
                 "model": OPENAI_MODEL_MINI,
                 "base_url": OPENAI_API_URL}

vanna_client = initialize_vanna(db_config=sqlite_config,
                                qdrant_config=qdrant_config,
                                openai_config=openai_config)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1927.11it/s]
/home/user/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/vanna/legacy/qdrant/qdrant.py:49: UserWarning: Api key is used with an insecure connection.
  self._client = QdrantClient(


In [13]:
q = """
Скрипт для выявления клиентов, у которых в любом 7-дневном окне сумма платежей заметно превышает их обычный уровень и средний показатель по стране проживания, с расчетом дневных сумм операций, скользящей 7-дневной суммы, базового среднего или медианы по предыдущим периодам, сравнением с 95-м перцентилем по стране клиента и дополнительной фиксацией случаев, когда в рамках окна платежи проводились через разных сотрудников или в разных магазинах; результат включает клиента, страну, город, период окна, сумму, количество платежей, число сотрудников, число магазинов и ранг подозрительности."
"""
r = vanna_client.generate_sql(question=q)

SQL Prompt: [{'role': 'system', 'content': "You are a SQLite expert. Please help to generate a SQL query to answer the question. Your response should ONLY be based on the given context and follow the response guidelines and format instructions. \n===Tables \nCREATE TABLE pay ( -- Таблица платежей клиентов за аренду. Содержит сумму, дату и связи с rental, customer, staff.\n  p01 int NOT NULL, -- Уникальный идентификатор платежа.\n  p02 INT NOT NULL, -- Идентификатор клиента, совершившего платёж. Внешний ключ к customer.\n  p03 SMALLINT NOT NULL, -- Идентификатор сотрудника, принявшего платёж. Внешний ключ к staff.\n  p04 INT DEFAULT NULL, -- Идентификатор аренды, за которую произведён платёж. Может быть NULL.\n  p05 DECIMAL(5,2) NOT NULL, -- Сумма платежа (до 2 знаков после запятой).\n  p06 TIMESTAMP NOT NULL, -- Дата и время платежа.\n  p07 TIMESTAMP NOT NULL, -- Дата последнего обновления записи платежа. Триггерная.\n  PRIMARY KEY (p01),\n  CONSTRAINT fk_pay_ren FOREIGN KEY (p04) REFE

In [14]:
vanna_client.run_sql(r)

,customer_id,customer_name,country_name,city_name,window_start,window_end,window_amount,payment_count,staff_count,store_count,customer_previous_avg_7d,country_p95_7d,suspicion_rank
0,181,ANA BRADLEY,United States,Memphis,2005-07-26,2005-08-01,82.86,14,2,2,18.97,41.91,1
1,459,TOMMY COLLAZO,Iran,Qomsheh,2005-07-26,2005-08-01,96.84,16,2,2,23.38,44.92,2
2,425,FRANCIS SIKES,Mexico,San Juan Bautista Tuxtepec,2005-07-02,2005-07-08,41.93,7,2,2,7.99,40.91,3
3,331,ERIC ROBERT,Argentina,Santa F,2005-07-01,2005-07-07,40.94,6,2,2,7.98,37.91,4
4,2,PATRICIA JOHNSON,United States,San Bernardino,2005-07-24,2005-07-30,62.89,11,2,2,13.37,41.91,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
570,573,BYRON BOX,Kazakstan,Zhezqazghan,2005-07-27,2005-08-02,34.92,8,2,2,17.17,34.92,571
571,517,BRAD MCCURDY,Taiwan,Chungho,2005-07-06,2005-07-12,41.90,10,2,2,20.62,41.90,572
572,332,STEPHEN QUALLS,Bangladesh,Dhaka,2005-07-27,2005-08-02,62.87,13,2,2,31.05,62.87,573
573,498,GENE SANBORN,Oman,Salala,2005-07-26,2005-08-01,31.92,8,2,2,15.85,31.92,574


In [15]:
script = """WITH customer_geo AS (
    SELECT
        c.h01 AS customer_id,
        c.h03 || ' ' || c.h04 AS customer_name,
        co.c02 AS country_name,
        ci.d02 AS city_name
    FROM cus AS c
    JOIN adr AS a ON a.e01 = c.h06
    JOIN cty AS ci ON ci.d01 = a.e05
    JOIN cnt AS co ON co.c01 = ci.d03
),
daily_payments AS (
    SELECT
        p.p02 AS customer_id,
        date(p.p06) AS payment_day,
        SUM(CAST(p.p05 AS REAL)) AS daily_amount,
        COUNT(*) AS daily_payment_count,
        COUNT(DISTINCT p.p03) AS daily_staff_count,
        COUNT(DISTINCT stf.o07) AS daily_store_count
    FROM pay AS p
    JOIN stf AS stf ON stf.o01 = p.p03
    GROUP BY
        p.p02,
        date(p.p06)
),
daily_base AS (
    SELECT
        dp.*,
        AVG(dp.daily_amount) OVER (
            PARTITION BY dp.customer_id
            ORDER BY julianday(dp.payment_day)
            ROWS BETWEEN 30 PRECEDING AND 1 PRECEDING
        ) AS prev_avg_30d,
        MEDIAN(dp.daily_amount) OVER (
            PARTITION BY dp.customer_id
            ORDER BY julianday(dp.payment_day)
            ROWS BETWEEN 30 PRECEDING AND 1 PRECEDING
        ) AS prev_median_30d
    FROM daily_payments AS dp
),
windows AS (
    SELECT
        d1.customer_id,
        d1.payment_day AS window_start,
        date(d1.payment_day, '+6 day') AS window_end,
        SUM(d2.daily_amount) AS window_sum,
        SUM(d2.daily_payment_count) AS window_payment_count,
        SUM(d2.daily_staff_count) AS window_staff_count_raw,
        SUM(d2.daily_store_count) AS window_store_count_raw,
        MAX(d2.prev_avg_30d) AS base_avg_30d,
        MAX(d2.prev_median_30d) AS base_median_30d
    FROM daily_base AS d1
    JOIN daily_base AS d2
      ON d2.customer_id = d1.customer_id
     AND d2.payment_day >= d1.payment_day
     AND d2.payment_day < date(d1.payment_day, '+7 day')
    GROUP BY
        d1.customer_id,
        d1.payment_day
),
country_p95 AS (
    SELECT
        cg.country_name,
        dp.payment_day,
        percentile_cont(0.95) WITHIN GROUP (ORDER BY dp.daily_amount) AS country_p95_amount
    FROM daily_payments AS dp
    JOIN customer_geo AS cg
      ON cg.customer_id = dp.customer_id
    GROUP BY
        cg.country_name,
        dp.payment_day
),
suspicious AS (
    SELECT
        w.customer_id,
        cg.customer_name,
        cg.country_name,
        cg.city_name,
        w.window_start,
        w.window_end,
        w.window_sum,
        w.window_payment_count,
        w.window_staff_count_raw AS staff_count,
        w.window_store_count_raw AS store_count,
        COALESCE(w.base_avg_30d, w.base_median_30d) AS baseline_value,
        cp.country_p95_amount,
        CASE
            WHEN COALESCE(w.base_avg_30d, w.base_median_30d) IS NULL OR COALESCE(w.base_avg_30d, w.base_median_30d) = 0 THEN NULL
            ELSE w.window_sum / COALESCE(w.base_avg_30d, w.base_median_30d)
        END AS suspicion_ratio,
        ROW_NUMBER() OVER (
            PARTITION BY cg.country_name
            ORDER BY
                (w.window_sum / NULLIF(COALESCE(w.base_avg_30d, w.base_median_30d), 0)) DESC,
                w.window_sum DESC
        ) AS suspicion_rank
    FROM windows AS w
    JOIN customer_geo AS cg
      ON cg.customer_id = w.customer_id
    LEFT JOIN country_p95 AS cp
      ON cp.country_name = cg.country_name
     AND cp.payment_day BETWEEN w.window_start AND w.window_end
    WHERE w.window_sum > COALESCE(w.base_avg_30d, w.base_median_30d) * 2
      AND w.window_sum > COALESCE(cp.country_p95_amount, 0)
      AND (w.window_staff_count_raw > 1 OR w.window_store_count_raw > 1)
)
SELECT
    customer_id,
    customer_name,
    country_name,
    city_name,
    window_start,
    window_end,
    ROUND(window_sum, 2) AS window_sum,
    window_payment_count,
    staff_count,
    store_count,
    ROUND(baseline_value, 2) AS baseline_value,
    ROUND(country_p95_amount, 2) AS country_p95_amount,
    ROUND(suspicion_ratio, 2) AS suspicion_ratio,
    suspicion_rank
FROM suspicious
ORDER BY suspicion_rank, window_sum DESC;"""

In [16]:
vanna_client.run_sql(script)

DatabaseError: Execution failed on sql 'WITH customer_geo AS (
    SELECT
        c.h01 AS customer_id,
        c.h03 || ' ' || c.h04 AS customer_name,
        co.c02 AS country_name,
        ci.d02 AS city_name
    FROM cus AS c
    JOIN adr AS a ON a.e01 = c.h06
    JOIN cty AS ci ON ci.d01 = a.e05
    JOIN cnt AS co ON co.c01 = ci.d03
),
daily_payments AS (
    SELECT
        p.p02 AS customer_id,
        date(p.p06) AS payment_day,
        SUM(CAST(p.p05 AS REAL)) AS daily_amount,
        COUNT(*) AS daily_payment_count,
        COUNT(DISTINCT p.p03) AS daily_staff_count,
        COUNT(DISTINCT stf.o07) AS daily_store_count
    FROM pay AS p
    JOIN stf AS stf ON stf.o01 = p.p03
    GROUP BY
        p.p02,
        date(p.p06)
),
daily_base AS (
    SELECT
        dp.*,
        AVG(dp.daily_amount) OVER (
            PARTITION BY dp.customer_id
            ORDER BY julianday(dp.payment_day)
            ROWS BETWEEN 30 PRECEDING AND 1 PRECEDING
        ) AS prev_avg_30d,
        MED